# Module 10.2: Speculative Decoding

As we established in previous modules, generating text with LLMs requires "autoregressive generation." You feed the prompt in, get the next token, append it, feed it ALL back in, get the next one, append it, feed it ALL back in. 

This means if we generate 100 new words, we must load the massive model weights into the GPU compute cores 100 separate times! This makes LLM inference **Memory Bandwidth Bound**, not Compute Bound. Our extremely fast GPU cores are spending 99% of their time waiting for the weights to be loaded from VRAM.

## 1. The Core Idea: The Manager and the Assistant

### The Analogy
Imagine a brilliant Software Architect (the 70B parameter Target Model) writing code. They spend 10 minutes thinking, then write 1 single line of code. They are perfect, but slow.

Now, imagine the Architect hires a Junior Developer (a 1B parameter Draft Model). The Junior Dev types incredibly fast but makes mistakes. 
The Junior Dev aggressively types out 5 lines of code in 10 seconds. 
The Architect then reads all 5 lines *at the same time* (in parallel). 
The Architect says: *"Line 1 is good, Line 2 is good, Line 3 is good, but Line 4 is a bug. Keep the first 3 lines, throw away the rest, and I will write Line 4 myself."*

**This is Speculative Decoding!**
- The **Draft Model** quickly predicts 5 future tokens.
- The **Target Model** evaluates all 5 tokens simultaneously in one single forward pass.
- If the fast model guessed right, we get multiple tokens for the "cost" of one model load. If not, we fall back instantly to the Target Model's output.

## 2. Implementing the Concept

Because evaluating a sequence of known tokens in parallel is mathematically identical to a normal prompt processing pass, the Target model can verify all guesses at once. Let's mock this out.

In [ ]:
import torch
import time

torch.manual_seed(0)  # Reproducible results

# Pretend we have a Vocabulary of 50,000 words
VOCAB_SIZE = 50000

# -----------------------------------------------------------------------------
# To DEMONSTRATE acceptances, both mock models share a hidden "ground truth"
# continuation. In a real system there is no such oracle -- the draft model is
# just a smaller network that happens to agree with the big model most of the
# time. We fake that agreement here so the concept is actually visible.
# -----------------------------------------------------------------------------

# The "true" sequence the Target Model will deterministically produce next,
# given any context. (Think of it as the article the big model wants to write.)
GROUND_TRUTH = torch.tensor([101, 202, 303, 404, 505, 606, 707, 808, 909, 111])

DRAFT_ACCURACY = 0.7  # Probability the fast draft model guesses each token right


def mock_target_model(sequence_tokens):
    """The huge, slow, but perfect model.

    Takes 0.1s to load its weights, but thanks to parallel attention it scores
    ALL positions of the sequence in that single pass. For position i it returns
    the token it would generate to come *after* sequence_tokens[i].

    We make it deterministic by reading from GROUND_TRUTH, indexed by how many
    tokens have been generated so far (i.e. position beyond the original prompt).
    """
    time.sleep(0.1)
    seq_len = sequence_tokens.size(0)
    # The number of already-generated (non-prompt) tokens currently in the seq.
    # PROMPT_LEN is a module-level constant set where we kick off the loop.
    preds = torch.empty(seq_len, dtype=torch.long)
    for i in range(seq_len):
        # Position i predicts the token that follows it. If position i sits at
        # generation-index g, the next token is GROUND_TRUTH[g].
        g = (i + 1) - PROMPT_LEN  # 0 for the first generated token, etc.
        if 0 <= g < len(GROUND_TRUTH):
            preds[i] = GROUND_TRUTH[g]
        else:
            preds[i] = 0  # padding / out of our scripted range
    return preds


def mock_draft_model(prompt_tokens, num_guesses=4):
    """The tiny, fast model. Takes 0.01s.

    It CONDITIONS on its input: it looks at how many tokens have been generated
    so far (len(prompt_tokens) - PROMPT_LEN) to know where it is in the sequence,
    then guesses the upcoming tokens. With probability DRAFT_ACCURACY it nails the
    same token the Target Model would pick; otherwise it emits a random wrong token.
    """
    time.sleep(0.01)
    generated_so_far = prompt_tokens.size(0) - PROMPT_LEN
    guesses = torch.empty(num_guesses, dtype=torch.long)
    for k in range(num_guesses):
        g = generated_so_far + k
        correct = GROUND_TRUTH[g] if g < len(GROUND_TRUTH) else torch.tensor(0)
        if torch.rand(1).item() < DRAFT_ACCURACY:
            guesses[k] = correct
        else:
            # A plausible-but-wrong token (anything that isn't the correct one)
            guesses[k] = torch.randint(0, VOCAB_SIZE, (1,)).item()
    return guesses


## 3. The Verification Logic

We compare the Junior's guesses against the Architect's true predictions, in one parallel Target pass.

### Which target position verifies which draft token?
The key insight: a model at position `i` predicts the token for position `i+1`. So after we append the drafts to the prompt, the Target Model's prediction at the *last prompt position* tells us the correct token for draft slot 0, and so on. Here is the layout for a prompt of length `L` with 4 drafts `d0..d3`:

```
index:      ...   L-1     L      L+1     L+2     L+3
token:      ...  prompt   d0      d1      d2      d3
                  │       │       │       │       │
target_preds:     ▼       ▼       ▼       ▼       ▼
                correct  correct correct correct  "next token
                for d0   for d1  for d2  for d3   after all drafts"
```

- `target_preds[L-1 : L+3]` (the slice of length `num_drafts`) are the **correct tokens for the drafts**. We compare each draft `d_k` to `target_preds[L-1+k]`.
- `target_preds[L+3]` (the very last) is the **bonus token**: what the Target Model wants after *all* drafts. We only get to use it if every single draft was accepted — that is the "all-accepted" branch, and it is what lets a perfect draft run yield `num_drafts + 1` tokens from one pass.
- We accept drafts left-to-right and stop at the first mismatch. At the first rejection we substitute the Target Model's own correct token, guaranteeing the output is identical to what the Target Model alone would have produced (greedy).

In [ ]:
def speculative_decode_step(current_seq, num_drafts=4):
    print(f"\n--- Speculative Step (context length {len(current_seq)}) ---")
    L = len(current_seq)

    # 1. Draft model guesses `num_drafts` tokens ahead (fast, conditioned on context)
    draft_guesses = mock_draft_model(current_seq, num_drafts)
    print(f"Draft guesses:            {draft_guesses.tolist()}")

    # 2. Append the drafts and run ONE parallel Target pass over the whole thing
    speculative_seq = torch.cat([current_seq, draft_guesses])
    target_preds = mock_target_model(speculative_seq)

    # 3. Pull out the predictions that verify each draft.
    #    Position (L-1) predicts draft 0, ..., position (L-1 + num_drafts - 1)
    #    predicts the last draft. (See the diagram above.)
    correct_for_drafts = target_preds[L - 1 : L - 1 + num_drafts]
    bonus_token        = target_preds[L - 1 + num_drafts]  # used only if ALL accepted
    print(f"Target's correct tokens:  {correct_for_drafts.tolist()}")

    # 4. Accept drafts left-to-right; stop at the first disagreement.
    accepted_count = 0
    for k in range(num_drafts):
        if draft_guesses[k].item() == correct_for_drafts[k].item():
            accepted_count += 1
        else:
            break

    accepted_tokens = draft_guesses[:accepted_count]

    # 5. Decide the extra Target-provided token.
    if accepted_count < num_drafts:
        # A draft was rejected -> substitute the Target's correct token there.
        # This is what guarantees identical-to-greedy output.
        correction = correct_for_drafts[accepted_count].unsqueeze(0)
        print(f"  -> Accepted {accepted_count} draft(s), then corrected token "
              f"{correction.item()} (draft said {draft_guesses[accepted_count].item()}).")
    else:
        # ALL-ACCEPTED branch: every draft matched, so we additionally take the
        # Target's "next" prediction, yielding num_drafts + 1 tokens from one pass.
        correction = bonus_token.unsqueeze(0)
        print(f"  -> Accepted ALL {accepted_count} drafts + 1 bonus token "
              f"{correction.item()} = {accepted_count + 1} tokens this pass!")

    final_yield = torch.cat([accepted_tokens, correction])
    print(f"Yielded {len(final_yield)} token(s) from 1 Target pass: {final_yield.tolist()}")
    return final_yield


# Kick off the loop. PROMPT_LEN is read by both mock models so they know where
# in the sequence we are (how many tokens have already been generated).
PROMPT_LEN = 4
prompt = torch.tensor([5, 12, 59, 1002])  # Mock prompt of length PROMPT_LEN

generated = 0
while generated < len(GROUND_TRUTH):
    new_tokens = speculative_decode_step(prompt, num_drafts=4)
    prompt = torch.cat([prompt, new_tokens])
    generated += len(new_tokens)

print(f"\nFull generated continuation: {prompt[PROMPT_LEN:].tolist()}")
print(f"Ground truth target wanted:  {GROUND_TRUTH.tolist()}")

## 4. Why does this accelerate LLMs?
If the Draft Model is moderately good (even just 40-50% accuracy on predicting the exact token), we average generating ~2-3 tokens per Target Model evaluation.

Because the Target Model is bottlenecked by memory bandwidth (loading weights), a forward pass for 1 token takes nearly the same wall-clock time as a forward pass for 5 tokens — the weights only load once either way. Thus, Speculative Decoding gives a 2x-3x speedup on inference.

### The output-quality caveat (important!)
This notebook does **greedy** decoding (always take the argmax). For greedy decoding, the verify-and-substitute rule above makes the output *bit-for-bit identical* to running the Target Model alone — we never keep a token the Target Model wouldn't have produced.

For **sampling** (temperature > 0, top-p, etc.) the simple "accept if it matches" check is *not* enough to preserve the distribution. Correct speculative *sampling* uses a probabilistic rule: accept a draft token with probability `min(1, p_target / p_draft)`, and on rejection resample from the corrected (renormalized) residual distribution. That correction is what keeps the sampled output statistically identical to sampling from the Target Model directly. We only implement the greedy case here for clarity.

## 5. Wall-clock comparison: speculative vs. naive one-token-at-a-time

Let's actually time it. The naive approach calls the slow Target Model once per token. Speculative decoding calls the (fast) draft once and the (slow) target once per *step*, but each step can yield several tokens.

In [ ]:
import io
from contextlib import redirect_stdout

TOKENS_TO_GENERATE = len(GROUND_TRUTH)

# --- Naive: one slow Target pass per token ---
PROMPT_LEN = 4
seq = torch.tensor([5, 12, 59, 1002])
t0 = time.time()
for _ in range(TOKENS_TO_GENERATE):
    preds = mock_target_model(seq)      # one full (slow) Target pass...
    next_tok = preds[-1].unsqueeze(0)   # ...for exactly ONE new token
    seq = torch.cat([seq, next_tok])
naive_time = time.time() - t0

# --- Speculative: one draft + one Target pass per step, multiple tokens/step ---
prompt = torch.tensor([5, 12, 59, 1002])
target_passes = 0
t0 = time.time()
generated = 0
with redirect_stdout(io.StringIO()):  # silence the per-step prints from above
    while generated < TOKENS_TO_GENERATE:
        new_tokens = speculative_decode_step(prompt, num_drafts=4)
        prompt = torch.cat([prompt, new_tokens])
        generated += len(new_tokens)
        target_passes += 1
spec_time = time.time() - t0

print(f"Generated {TOKENS_TO_GENERATE} tokens.\n")
print(f"Naive (1 token / Target pass): {naive_time:.3f}s  "
      f"({TOKENS_TO_GENERATE} Target passes)")
print(f"Speculative decoding:          {spec_time:.3f}s  "
      f"({target_passes} Target passes)")
print(f"\nWall-clock speedup: {naive_time / spec_time:.2f}x  "
      f"(fewer slow Target passes is the whole game)")

### 🏋️ Try it yourself

1. **Acceptance rate vs. speedup.** Loop over `DRAFT_ACCURACY` values `[0.3, 0.5, 0.7, 0.9]` (set the global, then re-run the speculative loop) and print the number of Target passes for each. Confirm that a smarter draft model means fewer expensive Target passes — and that a *bad* draft model can make speculative decoding barely better than naive.

2. **Does `num_drafts` always help?** Try `num_drafts` of 2, 4, and 8 at a fixed accuracy. More drafts means more tokens per *accepted* run, but every wrong guess past the first rejection is wasted work. Find the value that minimizes Target passes for your accuracy setting.

In [ ]:
# Starter code for Task 1 — measure Target passes at different draft accuracies.

def count_target_passes(draft_accuracy, num_drafts=4):
    global DRAFT_ACCURACY
    DRAFT_ACCURACY = draft_accuracy
    prompt = torch.tensor([5, 12, 59, 1002])
    passes, generated = 0, 0
    with redirect_stdout(io.StringIO()):  # hush the prints
        while generated < len(GROUND_TRUTH):
            new_tokens = speculative_decode_step(prompt, num_drafts=num_drafts)
            prompt = torch.cat([prompt, new_tokens])
            generated += len(new_tokens)
            passes += 1
    return passes

torch.manual_seed(0)
for acc in [0.3, 0.5, 0.7, 0.9]:
    p = count_target_passes(acc)
    print(f"Draft accuracy {acc:.1f} -> {p} Target passes "
          f"for {len(GROUND_TRUTH)} tokens")

# TODO (Task 2): wrap this in another loop over num_drafts in [2, 4, 8] and
#                find which minimizes Target passes at a fixed accuracy.